In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
%cd /content/drive/MyDrive

/content/drive/MyDrive


In [4]:
from pathlib import Path

print(Path("/content/drive/MyDrive/PACS/PACS/photo").exists())
print(Path("/content/drive/MyDrive/shared/splits/pacs_sketch_seed6304.json").exists())

True
True


In [5]:
%cd /content/drive/MyDrive

!python -m task2.train --config task2/configs/source_only.yaml

/content/drive/MyDrive
Device: cuda
photo: 1336 train, 334 validation
art_painting: 1638 train, 410 validation
cartoon: 1875 train, 469 validation
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 130MB/s]

Epoch 01 | Loss: 0.5251 | Mean Source Macro-F1: 0.8698
  photo         | Acc: 0.9461 | Macro-F1: 0.9342
  art_painting  | Acc: 0.7780 | Macro-F1: 0.7832
  cartoon       | Acc: 0.8785 | Macro-F1: 0.8920
  --> Best checkpoint saved: /content/drive/MyDrive/task2/results/source_only_erm_best.pth

Epoch 02 | Loss: 0.2580 | Mean Source Macro-F1: 0.9196
  photo         | Acc: 0.9551 | Macro-F1: 0.9455
  art_painting  | Acc: 0.8878 | Macro-F1: 0.8846
  cartoon       | Acc: 0.9190 | Macro-F1: 0.9286
  --> Best checkpoint saved: /content/drive/MyDrive/task2/results/source_only_erm_best.pth

Epoch 03 | Loss: 0.1729 | Mean Source Macro-F1: 0.9223
  photo         | Acc: 0.9671 | Macr

In [2]:
%cd /content/drive/MyDrive/task2
%cd /content/drive/MyDrive/shared

/content/drive/MyDrive/task2
/content/drive/MyDrive/shared


In [9]:
!touch shared/__init__.py
!touch task2/__init__.py

In [15]:
from datasets import load_dataset

dataset = load_dataset("Azeez577/PACS")

README.md:   0%|          | 0.00/1.74k [00:00<?, ?B/s]

PACS.zip: reconstructing file:   0%|          |  0.00B /  184MB            

PACS.zip: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/9991 [00:00<?, ? examples/s]

DatasetGenerationError: An error occurred while generating the dataset

In [16]:
from pathlib import Path

files = list(Path("/root/.cache/huggingface").rglob("PACS.zip"))
print(files)

[PosixPath('/root/.cache/huggingface/hub/datasets--Azeez577--PACS/snapshots/fb1ee820df83b4e9957251d9f6973eb6d424c494/PACS.zip')]


In [17]:
import zipfile
from pathlib import Path

zip_path = files[0]

extract_path = Path("/content/drive/MyDrive/PACS")
extract_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Extraction complete.")

Extraction complete.


In [18]:
from pathlib import Path

root = Path("/content/drive/MyDrive/PACS/PACS")

for domain in ["photo", "art_painting", "cartoon", "sketch"]:
    print(domain, (root / domain).exists())

photo True
art_painting True
cartoon True
sketch True


In [19]:
%cd /content/drive/MyDrive

!python -m shared.pacs_protocol \
    --data_root "/content/drive/MyDrive/PACS/PACS" \
    --output "shared/splits/pacs_sketch_seed6304.json"

/content/drive/MyDrive
photo: total=1670, train=1336, val=334
art_painting: total=2048, train=1638, val=410
cartoon: total=2344, train=1875, val=469

Saved split file to: shared/splits/pacs_sketch_seed6304.json


In [9]:
!pip install -q pyyaml

In [10]:
%cd /content/drive/MyDrive
!cat task2/configs/dan.yaml

/content/drive/MyDrive
seed: 6304

data:
  root: "/content/drive/MyDrive/PACS/PACS"
  split_file: "/content/drive/MyDrive/shared/splits/pacs_sketch_seed6304.json"

sources:
  - photo
  - art_painting
  - cartoon

target: sketch

model:
  num_classes: 7
  backbone: resnet18
  weights: IMAGENET1K_V1

training:
  epochs: 30
  patience: 5
  batch_size_per_source: 8
  target_batch_size: 24
  learning_rate: 0.0001
  weight_decay: 0.0001

adaptation:
  lambda: 1.0

method: dan

In [11]:
%cd /content/drive/MyDrive
!python -m task2.train --config task2/configs/dan.yaml

/content/drive/MyDrive
Device: cuda
Method: dan
photo: 1336 train, 334 validation
art_painting: 1638 train, 410 validation
cartoon: 1875 train, 469 validation

Epoch 01 | Total Loss: 0.5083 | Cls: 0.4541 | MMD: 0.0541 | Mean Source Macro-F1: 0.9138
  photo         | Acc: 0.9551 | Macro-F1: 0.9498
  art_painting  | Acc: 0.8561 | Macro-F1: 0.8533
  cartoon       | Acc: 0.9318 | Macro-F1: 0.9381
  --> Best checkpoint saved: /content/drive/MyDrive/task2/results/dan_best.pth

Epoch 02 | Total Loss: 0.2272 | Cls: 0.1988 | MMD: 0.0284 | Mean Source Macro-F1: 0.8966
  photo         | Acc: 0.9461 | Macro-F1: 0.9363
  art_painting  | Acc: 0.8707 | Macro-F1: 0.8587
  cartoon       | Acc: 0.8934 | Macro-F1: 0.8947

Epoch 03 | Total Loss: 0.1820 | Cls: 0.1591 | MMD: 0.0229 | Mean Source Macro-F1: 0.9264
  photo         | Acc: 0.9641 | Macro-F1: 0.9563
  art_painting  | Acc: 0.8854 | Macro-F1: 0.8865
  cartoon       | Acc: 0.9318 | Macro-F1: 0.9364
  --> Best checkpoint saved: /content/drive/MyDrive

In [4]:
%cd /content/drive/MyDrive

!python -m task2.train --config task2/configs/dann.yaml

/content/drive/MyDrive
Device: cuda
Method: dann
photo: 1336 train, 334 validation
art_painting: 1638 train, 410 validation
cartoon: 1875 train, 469 validation
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 136MB/s]

Epoch 01 | Total Loss: 820.8470 | Cls: 48.2832 | Domain: 772.5638 | Alpha: 0.0826 | Mean Source Macro-F1: 0.0585
  photo         | Acc: 0.2575 | Macro-F1: 0.0586
  art_painting  | Acc: 0.2195 | Macro-F1: 0.0518
  cartoon       | Acc: 0.1727 | Macro-F1: 0.0650
  --> Best checkpoint saved: /content/drive/MyDrive/task2/results/dann_best.pth

Epoch 02 | Total Loss: 118.9719 | Cls: 21.6838 | Domain: 97.2881 | Alpha: 0.2441 | Mean Source Macro-F1: 0.0351
  photo         | Acc: 0.1198 | Macro-F1: 0.0306
  art_painting  | Acc: 0.1000 | Macro-F1: 0.0286
  cartoon       | Acc: 0.1407 | Macro-F1: 0.0461

Epoch 03 | Total Loss: 623.8127 | Cls: 184.2827 | Domain: 439.530

In [6]:
%cd /content/drive/MyDrive
!python -m task2.train --config task2/configs/cdan.yaml

/content/drive/MyDrive
Device: cuda
Method: cdan
photo: 1336 train, 334 validation
art_painting: 1638 train, 410 validation
cartoon: 1875 train, 469 validation

Epoch 01 | Total Loss: 0.9723 | Cls: 0.5202 | Domain: 0.4521 | Alpha: 0.0826 | Mean Source Macro-F1: 0.8639
  photo         | Acc: 0.9222 | Macro-F1: 0.9076
  art_painting  | Acc: 0.8220 | Macro-F1: 0.8168
  cartoon       | Acc: 0.8635 | Macro-F1: 0.8671
  --> Best checkpoint saved: /content/drive/MyDrive/task2/results/cdan_best.pth

Epoch 02 | Total Loss: 0.8984 | Cls: 0.2630 | Domain: 0.6355 | Alpha: 0.2441 | Mean Source Macro-F1: 0.9026
  photo         | Acc: 0.9521 | Macro-F1: 0.9434
  art_painting  | Acc: 0.8341 | Macro-F1: 0.8418
  cartoon       | Acc: 0.9126 | Macro-F1: 0.9226
  --> Best checkpoint saved: /content/drive/MyDrive/task2/results/cdan_best.pth

Epoch 03 | Total Loss: 1.1111 | Cls: 0.3466 | Domain: 0.7645 | Alpha: 0.3931 | Mean Source Macro-F1: 0.9045
  photo         | Acc: 0.9551 | Macro-F1: 0.9487
  art_pain

In [11]:
%cd /content/drive/MyDrive
!python -m task2.evaluate_final

/content/drive/MyDrive
Device: cuda

Source-only
photo         | Acc: 0.9701 | Macro-F1: 0.9645
art_painting  | Acc: 0.8878 | Macro-F1: 0.8870
cartoon       | Acc: 0.9403 | Macro-F1: 0.9480

Mean Source Val | Acc: 0.9327 | Macro-F1: 0.9332
Target          | Acc: 0.5602 | Macro-F1: 0.5559

DAN
photo         | Acc: 0.9760 | Macro-F1: 0.9723
art_painting  | Acc: 0.9073 | Macro-F1: 0.9101
cartoon       | Acc: 0.9424 | Macro-F1: 0.9469

Mean Source Val | Acc: 0.9419 | Macro-F1: 0.9431
Target          | Acc: 0.7111 | Macro-F1: 0.6672

DANN
photo         | Acc: 0.2575 | Macro-F1: 0.0586
art_painting  | Acc: 0.2195 | Macro-F1: 0.0518
cartoon       | Acc: 0.1727 | Macro-F1: 0.0650

Mean Source Val | Acc: 0.2166 | Macro-F1: 0.0585
Target          | Acc: 0.0494 | Macro-F1: 0.0450

CDAN
photo         | Acc: 0.9641 | Macro-F1: 0.9572
art_painting  | Acc: 0.8585 | Macro-F1: 0.8615
cartoon       | Acc: 0.9318 | Macro-F1: 0.9377

Mean Source Val | Acc: 0.9181 | Macro-F1: 0.9188
Target          | Acc: 

In [12]:
%cd /content/drive/MyDrive
!python -m task2.train --config task2/configs/dan_lambda_0.1.yaml

/content/drive/MyDrive
Device: cuda
Method: dan
photo: 1336 train, 334 validation
art_painting: 1638 train, 410 validation
cartoon: 1875 train, 469 validation

Epoch 01 | Total Loss: 0.4743 | Cls: 0.4589 | MMD: 0.1543 | Mean Source Macro-F1: 0.9154
  photo         | Acc: 0.9611 | Macro-F1: 0.9521
  art_painting  | Acc: 0.8634 | Macro-F1: 0.8631
  cartoon       | Acc: 0.9254 | Macro-F1: 0.9310
  --> Best checkpoint saved: /content/drive/MyDrive/task2/results/dan_lambda_0.1_best.pth

Epoch 02 | Total Loss: 0.2040 | Cls: 0.1967 | MMD: 0.0731 | Mean Source Macro-F1: 0.9093
  photo         | Acc: 0.9551 | Macro-F1: 0.9478
  art_painting  | Acc: 0.8634 | Macro-F1: 0.8630
  cartoon       | Acc: 0.9083 | Macro-F1: 0.9172

Epoch 03 | Total Loss: 0.1416 | Cls: 0.1359 | MMD: 0.0563 | Mean Source Macro-F1: 0.9451
  photo         | Acc: 0.9671 | Macro-F1: 0.9613
  art_painting  | Acc: 0.9122 | Macro-F1: 0.9115
  cartoon       | Acc: 0.9574 | Macro-F1: 0.9626
  --> Best checkpoint saved: /content/dr

In [13]:
%cd /content/drive/MyDrive
!python -m task2.train --config task2/configs/dan_lambda_1.yaml

/content/drive/MyDrive
Device: cuda
Method: dan
photo: 1336 train, 334 validation
art_painting: 1638 train, 410 validation
cartoon: 1875 train, 469 validation

Epoch 01 | Total Loss: 0.5083 | Cls: 0.4541 | MMD: 0.0541 | Mean Source Macro-F1: 0.9138
  photo         | Acc: 0.9551 | Macro-F1: 0.9498
  art_painting  | Acc: 0.8561 | Macro-F1: 0.8533
  cartoon       | Acc: 0.9318 | Macro-F1: 0.9381
  --> Best checkpoint saved: /content/drive/MyDrive/task2/results/dan_lambda_1.0_best.pth

Epoch 02 | Total Loss: 0.2272 | Cls: 0.1988 | MMD: 0.0284 | Mean Source Macro-F1: 0.8966
  photo         | Acc: 0.9461 | Macro-F1: 0.9363
  art_painting  | Acc: 0.8707 | Macro-F1: 0.8587
  cartoon       | Acc: 0.8934 | Macro-F1: 0.8947

Epoch 03 | Total Loss: 0.1820 | Cls: 0.1591 | MMD: 0.0229 | Mean Source Macro-F1: 0.9264
  photo         | Acc: 0.9641 | Macro-F1: 0.9563
  art_painting  | Acc: 0.8854 | Macro-F1: 0.8865
  cartoon       | Acc: 0.9318 | Macro-F1: 0.9364
  --> Best checkpoint saved: /content/dr

In [14]:
%cd /content/drive/MyDrive
!python -m task2.train --config task2/configs/dan_lambda_10.yaml

/content/drive/MyDrive
Device: cuda
Method: dan
photo: 1336 train, 334 validation
art_painting: 1638 train, 410 validation
cartoon: 1875 train, 469 validation

Epoch 01 | Total Loss: 1.1114 | Cls: 0.7102 | MMD: 0.0401 | Mean Source Macro-F1: 0.8974
  photo         | Acc: 0.9581 | Macro-F1: 0.9504
  art_painting  | Acc: 0.8293 | Macro-F1: 0.8271
  cartoon       | Acc: 0.9104 | Macro-F1: 0.9146
  --> Best checkpoint saved: /content/drive/MyDrive/task2/results/dan_lambda_10.0_best.pth

Epoch 02 | Total Loss: 0.5437 | Cls: 0.3913 | MMD: 0.0152 | Mean Source Macro-F1: 0.8841
  photo         | Acc: 0.9341 | Macro-F1: 0.9220
  art_painting  | Acc: 0.8366 | Macro-F1: 0.8339
  cartoon       | Acc: 0.8955 | Macro-F1: 0.8964

Epoch 03 | Total Loss: 0.4353 | Cls: 0.2900 | MMD: 0.0145 | Mean Source Macro-F1: 0.8382
  photo         | Acc: 0.9102 | Macro-F1: 0.8992
  art_painting  | Acc: 0.7756 | Macro-F1: 0.7775
  cartoon       | Acc: 0.8316 | Macro-F1: 0.8379

Epoch 04 | Total Loss: 0.3098 | Cls: 0

In [15]:
%cd /content/drive/MyDrive
!python -m task2.evaluate_sensitivity

/content/drive/MyDrive

DAN LAMBDA SENSITIVITY STUDY

DAN λ=0.1
Source Acc: 0.9455
Source Macro-F1: 0.9451
Target Acc: 0.7674
Target Macro-F1: 0.7529
Domain Separability: 0.9884

DAN λ=1
Source Acc: 0.9419
Source Macro-F1: 0.9431
Target Acc: 0.7111
Target Macro-F1: 0.6672
Domain Separability: 0.9518

DAN λ=10
Source Acc: 0.8993
Source Macro-F1: 0.8974
Target Acc: 0.6396
Target Macro-F1: 0.6039
Domain Separability: 0.9219

DAN LAMBDA SENSITIVITY RESULTS
Setting             Src Acc      Src F1    Target Acc     Target F1   Domain Sep.
DAN λ=0.1            0.9455      0.9451        0.7674        0.7529        0.9884
DAN λ=1              0.9419      0.9431        0.7111        0.6672        0.9518
DAN λ=10             0.8993      0.8974        0.6396        0.6039        0.9219
